In [1]:
# Instalación de librerías necesarias
!pip install kmodes joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score
from kmodes.kprototypes import KPrototypes

In [2]:
# Base URL del repositorio en GitHub (carpeta Semana_6)
# Nota: Si tu rama principal es 'main', cámbialo por 'main'
BRANCH = "main"  # o "master"
BASE_URL = f"https://raw.githubusercontent.com/WalterVillatoro1204/marketing-analytics-walter-villatoro/{BRANCH}/Semana_6"

# Rutas completas a los archivos
url_clientes = f"{BASE_URL}/clientes_retail.csv"
url_lunas = f"{BASE_URL}/formas_lunas.csv"
url_alargadas = f"{BASE_URL}/formas_alargadas.csv"

In [3]:
df_alargadas = pd.read_csv(url_alargadas)
X_alargadas = df_alargadas[['x', 'y']]

# 1. Ajuste K-Means (asume grupos esféricos)
km_alargadas = KMeans(n_clusters=3, random_state=42, n_init=10)
df_alargadas['kmeans'] = km_alargadas.fit_predict(X_alargadas)

# 2. Ajuste Gaussian Mixture (se adapta a elipses/covarianza completa)
gmm_alargadas = GaussianMixture(n_components=3, covariance_type='full', random_state=42)
df_alargadas['gmm'] = gmm_alargadas.fit_predict(X_alargadas)

# Comparación gráfica
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.scatterplot(data=df_alargadas, x='x', y='y', hue='grupo_real', palette='tab10', ax=axes[0]).set_title('Real')
sns.scatterplot(data=df_alargadas, x='x', y='y', hue='kmeans', palette='tab10', ax=axes[1]).set_title('K-Means')
sns.scatterplot(data=df_alargadas, x='x', y='y', hue='gmm', palette='tab10', ax=axes[2]).set_title('GMM (Mezclas Gaussianas)')
plt.show()

# Puntos "con duda": sin ninguna probabilidad posterior > 0.90
probabilidades = gmm_alargadas.predict_proba(X_alargadas)
max_prob = probabilidades.max(axis=1)
puntos_con_duda = np.sum(max_prob <= 0.90)
print(f"Puntos sintéticos con duda (probabilidad max <= 0.90): {puntos_con_duda}")

HTTPError: HTTP Error 404: Not Found

In [ ]:
df_clientes = pd.read_csv(url_clientes)

# Variables numéricas del laboratorio
cols_num = [
    'ingreso_anual', 'gasto_total', 'compras_tienda',
    'compras_web', 'compras_catalogo', 'visitas_web_mes',
    'recencia_dias', 'hijos_pequenos', 'edad'
]
cols_cat = ['educacion', 'estado_civil']

# 1. Estandarización numérica
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(df_clientes[cols_num])

# 2. Segmentación K-Means (k=3)
kmeans_model = KMeans(n_clusters=3, random_state=42, n_init=10)
df_clientes['seg_kmeans'] = kmeans_model.fit_predict(X_num_scaled)

# 3. Segmentación GMM (k=3)
gmm_model = GaussianMixture(n_components=3, covariance_type='full', random_state=42)
df_clientes['seg_gmm'] = gmm_model.fit_predict(X_num_scaled)

# 4. Segmentación K-Prototypes (k=3, numéricas + categóricas)
# Preparamos la matriz combinando las numéricas escaladas y las categóricas
df_kproto_input = pd.DataFrame(X_num_scaled, columns=cols_num)
for c in cols_cat:
    df_kproto_input[c] = df_clientes[c].astype(str)

kproto_model = KPrototypes(n_clusters=3, init='Cao', random_state=42, n_jobs=-1)
# Los índices categóricos corresponden a las últimas dos columnas
cat_indices = [df_kproto_input.columns.get_loc(c) for c in cols_cat]
df_clientes['seg_kproto'] = kproto_model.fit_predict(df_kproto_input, categorical=cat_indices)

In [ ]:
# Concordancia entre K-Means y K-Prototypes (1.0 = idénticos)
ari_score = adjusted_rand_score(df_clientes['seg_kmeans'], df_clientes['seg_kproto'])
print(f"Acuerdo (Adjusted Rand Index) entre K-Means y K-Prototypes: {ari_score:.4f}")

# Matriz / Tabla cruzada
tabla_cruzada = pd.crosstab(
    df_clientes['seg_kmeans'],
    df_clientes['seg_kproto'],
    rownames=['K-Means'],
    colnames=['K-Prototypes']
)
print("\nTabla cruzada de asignación de clientes:")
print(tabla_cruzada)

# Comparar gasto medio por segmento en cada técnica
print("\nGasto medio por segmento:")
print("K-Means:\n", df_clientes.groupby('seg_kmeans')['gasto_total'].mean())
print("K-Prototypes:\n", df_clientes.groupby('seg_kproto')['gasto_total'].mean())

In [ ]:
# Mapeo a etiquetas estandarizadas C1, C2, C3
for col in ['seg_kmeans', 'seg_gmm', 'seg_kproto']:
    df_clientes[col] = df_clientes[col].map({0: 'C1', 1: 'C2', 2: 'C3'})

# Columna final recomendada para la app (por defecto k-prototypes)
df_clientes['Cluster'] = df_clientes['seg_kproto']

# 1. Guardar CSV para el visor de la aplicación Streamlit
df_clientes.to_csv('segmentos_notebook.csv', index=False)
# Si tu app.py busca clientes_con_cluster.csv, guardamos copia de respaldo:
df_clientes.to_csv('clientes_con_cluster.csv', index=False)

# 2. Guardar modelo GMM
joblib.dump({
    'modelo': gmm_model,
    'scaler': scaler,
    'features': cols_num
}, 'modelo_gmm.joblib')

# 3. Guardar modelo K-Prototypes
joblib.dump({
    'modelo': kproto_model,
    'scaler': scaler,
    'num_features': cols_num,
    'cat_features': cols_cat
}, 'modelo_kproto.joblib')

print("Archivos exportados exitosamente: segmentos_notebook.csv, modelo_gmm.joblib y modelo_kproto.joblib")